# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

### Step 1: Loading and Inspecting the Data

The first step is to load the aviation accident dataset into a Pandas DataFrame and inspect its structure before making any changes.

We will examine the number of rows and columns, column names, data types, missing values, duplicate records, and summary statistics.

This initial inspection helps us understand the quality of the data and identify columns that require cleaning before analysis.

In [87]:
# Locate the datasets

data_path = Path("data")

print("Files found:")
for file in data_path.iterdir():
    print(file.name)

Files found:
.gitkeep
AviationData.csv
USState_Codes.csv


In [88]:
# Load the aviation datasets

aviation = pd.read_csv(
    data_path / "AviationData.csv",
    encoding="ISO-8859-1",
    low_memory=False
)
aviation.head(5)

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [89]:
# Load the states datasets

states = pd.read_csv(
    data_path / "USState_Codes.csv",
    encoding="ISO-8859-1",
    low_memory=False
)
states.head(5)

,US_State,Abbreviation
0,Alabama,AL
1,Alaska,AK
2,Arizona,AZ
3,Arkansas,AR
4,California,CA


In [90]:
# Inspect the datasets
print("Aviation dataset shape:", aviation.shape)
print("\nState codes dataset shape:", states.shape)

Aviation dataset shape: (88889, 31)

State codes dataset shape: (62, 2)


In [91]:
# Summary statistics
aviation.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [92]:
# Inspect summary statistics
display(aviation.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
# Check for exact duplicate rows
print("Duplicate rows:", aviation.duplicated().sum())

Duplicate rows: 0


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [94]:
# Inspect all column names
print(aviation.columns.tolist())

# Identify important columns available in the dataset
relevant_columns = [
    "Event.Date",
    "Make",
    "Model",
    "Aircraft.Category",
    "Aircraft.Damage",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

available_columns = [
    col for col in relevant_columns
    if col in aviation.columns
]

display(aviation[available_columns].head())

# Inspect aircraft categories
if "Aircraft.Category" in aviation.columns:
    display(
        aviation["Aircraft.Category"]
        .value_counts(dropna=False)
    )

['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']


,Event.Date,Make,Model,Aircraft.Category,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
0,1948-10-24,Stinson,108-3,NaN,2.0,0.0,0.0,0.0
1,1962-07-19,Piper,PA24-180,NaN,4.0,0.0,0.0,0.0
2,1974-08-30,Cessna,172M,NaN,3.0,NaN,NaN,NaN
3,1977-06-19,Rockwell,112,NaN,2.0,0.0,0.0,0.0
4,1979-08-02,Cessna,501,NaN,1.0,2.0,NaN,0.0


Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [95]:
# Convert accident dates to datetime
aviation["Event.Date"] = pd.to_datetime(
    aviation["Event.Date"],
    errors="coerce"
)

# Examine invalid or missing dates
print(
    "Missing or invalid dates:",
    aviation["Event.Date"].isna().sum()
)

# Keep accidents occurring from 1983 onward
aviation = aviation.loc[
    aviation["Event.Date"].ge("1983-01-01")
].copy()

print("Records after date filtering:", aviation.shape)

Missing or invalid dates: 0
Records after date filtering: (85289, 31)


In [96]:
# Investigate missing values in relevant columns
relevant_columns = [
    col for col in relevant_columns
    if col in aviation.columns
]

display(
    aviation[relevant_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Aircraft.Category         56566
Total.Serious.Injuries    12481
Total.Minor.Injuries      11905
Total.Fatal.Injuries      11376
Total.Uninjured            5903
Model                        78
Make                         57
Event.Date                    0
dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [97]:
# Clean injury counts
injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

# Keep only columns that exist in the dataset
injury_columns = [
    col for col in injury_columns
    if col in aviation.columns
]

# Convert injury counts to numeric
for col in injury_columns:
    aviation[col] = pd.to_numeric(
        aviation[col],
        errors="coerce"
    )

# Inspect the cleaned injury columns
display(aviation[injury_columns].describe())

# Count missing values
display(aviation[injury_columns].isna().sum())

,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,73913.000000,72808.000000,73384.000000,79386.000000
mean,0.657530,0.283595,0.360828,5.461051
std,5.576903,1.574753,2.275808,28.412117
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,1.000000
75%,0.000000,0.000000,0.000000,2.000000
max,349.000000,161.000000,380.000000,699.000000


Total.Fatal.Injuries      11376
Total.Serious.Injuries    12481
Total.Minor.Injuries      11905
Total.Uninjured            5903
dtype: int64

In [98]:
# Identify invalid counts
for col in injury_columns:
    invalid_count = (
        aviation[col].notna()
        & aviation[col].lt(0)
    ).sum()

    print(f"{col}: {invalid_count} negative values")

for col in injury_columns:
    aviation.loc[
        aviation[col] < 0,
        col
    ] = np.nan

Total.Fatal.Injuries: 0 negative values
Total.Serious.Injuries: 0 negative values
Total.Minor.Injuries: 0 negative values
Total.Uninjured: 0 negative values


In [99]:
# Create derived injury measures
required_injury_columns = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

missing_required = [
    col for col in required_injury_columns
    if col not in aviation.columns
]

if missing_required:
    raise ValueError(
        f"Missing required injury columns: {missing_required}"
    )

# Estimate the total number of people aboard only when
# all four injury-count fields are available.
aviation["Total.People"] = (
    aviation[required_injury_columns]
    .sum(axis=1, min_count=4)
)

# Combined fatal and serious injury count
aviation["Total.Fatal.Serious.Injuries"] = (
    aviation[
        [
            "Total.Fatal.Injuries",
            "Total.Serious.Injuries"
        ]
    ].sum(axis=1, min_count=2)
)

# Proportion of people aboard who sustained fatal or serious injuries
aviation["Fatal.Serious.Injury.Rate"] = (
    aviation["Total.Fatal.Serious.Injuries"]
    / aviation["Total.People"].where(
        aviation["Total.People"] > 0
    )
)

# Inspect the new columns
display(
    aviation[
        [
            "Total.People",
            "Total.Fatal.Serious.Injuries",
            "Fatal.Serious.Injury.Rate"
        ]
    ].describe()
)

,Total.People,Total.Fatal.Serious.Injuries,Fatal.Serious.Injury.Rate
count,70793.000000,71262.000000,69710.000000
mean,5.975633,0.806138,0.292633
std,27.246370,5.116804,0.437875
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000
max,576.000000,295.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [100]:
# Inspect the original damage categories
# Inspect the original damage categories
display(
    aviation["Aircraft.damage"]
    .value_counts(dropna=False)
)

aviation["Aircraft.damage"] = (
    aviation["Aircraft.damage"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Standardize blank values
aviation["Aircraft.damage"] = (
    aviation["Aircraft.damage"]
    .replace("", pd.NA)
)

display(
    aviation["Aircraft.damage"]
    .value_counts(dropna=False)
)

known_damage = (
    aviation["Aircraft.damage"].notna()
    & aviation["Aircraft.damage"].ne("UNKNOWN")
)

aviation["Aircraft.Destroyed"] = pd.Series(
    pd.NA,
    index=aviation.index,
    dtype="Int64"
)

aviation.loc[
    known_damage,
    "Aircraft.Destroyed"
] = (
    aviation.loc[
        known_damage,
        "Aircraft.damage"
    ].eq("DESTROYED").astype(int)
)

display(
    aviation[
        ["Aircraft.damage", "Aircraft.Destroyed"]
    ].head(10)
)

display(
    aviation["Aircraft.Destroyed"]
    .value_counts(dropna=False)
)


Aircraft.damage
Substantial    61775
Destroyed      17575
NaN             3138
Minor           2682
Unknown          119
Name: count, dtype: int64

Aircraft.damage
SUBSTANTIAL    61775
DESTROYED      17575
<NA>            3138
MINOR           2682
UNKNOWN          119
Name: count, dtype: Int64

,Aircraft.damage,Aircraft.Destroyed
3600,<NA>,<NA>
3601,SUBSTANTIAL,0
3602,SUBSTANTIAL,0
3603,SUBSTANTIAL,0
3604,SUBSTANTIAL,0
3605,SUBSTANTIAL,0
3606,<NA>,<NA>
3607,DESTROYED,1
3608,DESTROYED,1
3609,DESTROYED,1


Aircraft.Destroyed
0       64457
1       17575
<NA>     3257
Name: count, dtype: Int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [101]:
# Inspect missing values
print("Missing makes:", aviation["Make"].isna().sum())

# Inspect the most frequent makes
display(
    aviation["Make"]
    .value_counts(dropna=False)
    .head(30)
)

# Standardize whitespace and capitalization
aviation["Make"] = (
    aviation["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Replace blank strings with missing values
aviation["Make"] = aviation["Make"].replace(
    "",
    pd.NA
)

# Inspect the updated values
display(
    aviation["Make"]
    .value_counts(dropna=False)
    .head(30)
)

Missing makes: 57


Make
Cessna               20892
Piper                11301
CESSNA                4922
Beech                 4067
PIPER                 2841
Bell                  2022
Boeing                1545
BOEING                1151
BEECH                 1042
Mooney                1036
Grumman                990
Robinson               923
Bellanca               824
Hughes                 742
Schweizer              612
BELL                   588
Air Tractor            577
Mcdonnell Douglas      521
Aeronca                458
Maule                  428
Champion               416
De Havilland           367
Aero Commander         343
Stinson                330
Aerospatiale           316
Rockwell               315
Taylorcraft            304
Luscombe               296
Hiller                 295
North American         291
Name: count, dtype: int64

Make
CESSNA               25814
PIPER                14142
BEECH                 5109
BOEING                2696
BELL                  2610
MOONEY                1278
ROBINSON              1207
GRUMMAN               1068
BELLANCA               983
HUGHES                 879
SCHWEIZER              756
AIR TRACTOR            673
AERONCA                607
MCDONNELL DOUGLAS      603
MAULE                  572
CHAMPION               507
STINSON                421
AERO COMMANDER         412
DE HAVILLAND           407
LUSCOMBE               391
NORTH AMERICAN         370
TAYLORCRAFT            366
AEROSPATIALE           362
ROCKWELL               339
HILLER                 332
AIRBUS                 291
ENSTROM                281
DOUGLAS                267
EMBRAER                236
AYRES                  234
Name: count, dtype: Int64

In [102]:
# Filter makes with at least 50 accident records
make_counts = aviation["Make"].value_counts()

eligible_makes = make_counts[
    make_counts >= 50
].index

aviation = aviation.loc[
    aviation["Make"].isin(eligible_makes)
].copy()

print("Dataset shape after make filtering:", aviation.shape)

print(
    "Number of eligible makes:",
    aviation["Make"].nunique()
)

Dataset shape after make filtering: (71340, 35)
Number of eligible makes: 97


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [103]:
# Inspect missing model values
print("Missing models:", aviation["Model"].isna().sum())

# Inspect the most frequent model labels
display(
    aviation["Model"]
    .value_counts(dropna=False)
    .head(30)
)

# Standardize model labels
aviation["Model"] = (
    aviation["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
)

aviation["Model"] = aviation["Model"].replace(
    "",
    pd.NA
)

# Remove records without a model
aviation = aviation.dropna(
    subset=["Model"]
).copy()

print("Dataset shape after model cleaning:", aviation.shape)

Missing models: 23


Model
152          2231
172          1654
172N         1096
PA-28-140     869
172M          760
150           752
172P          665
182           618
180           596
150M          553
PA-18-150     550
PA-18         550
PA-28-180     548
PA-28-161     526
PA-28-181     509
737           489
206B          488
150L          436
A36           428
PA-38-112     426
G-164A        423
140           385
170B          374
206           374
172S          371
G-164B        366
R44           360
PA-32-300     337
182P          333
PA-24-250     332
Name: count, dtype: int64

Dataset shape after model cleaning: (71317, 35)


In [104]:
# Check whether model labels are unique to each make
# Count records by make/model

make_model_counts = (
    aviation.groupby(
        ["Make", "Model"],
        dropna=False
    )
    .size()
    .reset_index(name="Accident.Count")
    .sort_values("Accident.Count", ascending=False)
)

display(make_model_counts.head(20))

# Count how many distinct makes use each model label
model_make_counts = (
    aviation.groupby("Model")["Make"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Models associated with multiple makes:",
    (model_make_counts > 1).sum()
)

display(
    model_make_counts[
        model_make_counts > 1
    ].head(20)
)

,Make,Model,Accident.Count
2523,CESSNA,152,2231
2546,CESSNA,172,1654
2597,CESSNA,172N,1096
5449,PIPER,PA-28-140,869
2595,CESSNA,172M,760
2496,CESSNA,150,752
2600,CESSNA,172P,665
2652,CESSNA,182,618
2629,CESSNA,180,595
2522,CESSNA,150M,553


Models associated with multiple makes: 480


Model
500       7
G-164A    7
G-164B    6
AA-5      5
AA-5B     5
G-164     5
G-164C    5
7ECA      4
500-S     4
269C      4
AA-1B     4
690A      4
100       4
112       4
690B      4
690C      4
AA-1C     4
AA1B      4
700       4
8KCAB     4
Name: Make, dtype: int64

In [105]:
# Create a unique aircraft-type identifier
aviation["Make.Model"] = (
    aviation["Make"].astype("string")
    + " "
    + aviation["Model"].astype("string")
)

display(
    aviation[
        ["Make", "Model", "Make.Model"]
    ].head(10)
)

,Make,Model,Make.Model
3601,CESSNA,182P,CESSNA 182P
3602,CESSNA,182RG,CESSNA 182RG
3603,CESSNA,182P,CESSNA 182P
3604,PIPER,PA-28R-200,PIPER PA-28R-200
3605,CESSNA,140,CESSNA 140
3606,BALLOON WORKS,FIREFLY 7B,BALLOON WORKS FIREFLY 7B
3607,CESSNA,340A,CESSNA 340A
3608,NORTH AMERICAN,T-6G,NORTH AMERICAN T-6G
3609,PIPER,PA-24-250,PIPER PA-24-250
3610,PIPER,PA-32-301R,PIPER PA-32-301R


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [106]:
# Inspect categories and missing values
other_columns = [
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

other_columns = [
    col for col in other_columns
    if col in aviation.columns
]

for col in other_columns:
    print(f"\n--- {col} ---")
    print("Missing values:", aviation[col].isna().sum())
    display(aviation[col].value_counts(dropna=False).head(30))


--- Engine.Type ---
Missing values: 5586


Engine.Type
Reciprocating      55620
NaN                 5586
Turbo Shaft         2997
Turbo Prop          2802
Turbo Fan           2184
Unknown             1568
Turbo Jet            545
Geared Turbofan       12
LR                     1
UNK                    1
NONE                   1
Name: count, dtype: int64


--- Weather.Condition ---
Missing values: 3807


Weather.Condition
VMC    61251
IMC     5285
NaN     3807
UNK      743
Unk      231
Name: count, dtype: int64


--- Number.of.Engines ---
Missing values: 5005


Number.of.Engines
1.0    55076
2.0     9649
NaN     5005
0.0      753
3.0      432
4.0      401
8.0        1
Name: count, dtype: int64


--- Purpose.of.flight ---
Missing values: 5437


Purpose.of.flight
Personal                     37456
Instructional                 9440
Unknown                       5566
NaN                           5437
Aerial Application            4144
Business                      3457
Positioning                   1451
Other Work Use                1069
Aerial Observation             713
Public Aircraft                662
Ferry                          645
Executive/corporate            430
Skydiving                      175
Flight Test                    165
Banner Tow                      97
External Load                   85
Public Aircraft - Federal       76
Public Aircraft - State         58
Public Aircraft - Local         52
Glider Tow                      41
Firefighting                    33
Air Race show                   30
Air Race/show                   18
Air Drop                        10
PUBS                             3
ASHO                             3
PUBL                             1
Name: count, dtype: int64


--- Broad.phase.of.flight ---
Missing values: 20849


Broad.phase.of.flight
NaN            20849
Landing        13105
Takeoff         9869
Cruise          8462
Maneuvering     6347
Approach        5256
Taxi            1678
Climb           1666
Descent         1573
Go-around       1192
Standing         834
Unknown          403
Other             83
Name: count, dtype: int64

In [107]:
# Clean categorical columns
categorical_columns = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight"
]

for col in categorical_columns:
    if col in aviation.columns:
        aviation[col] = (
            aviation[col]
            .astype("string")
            .str.strip()
        )

        aviation[col] = aviation[col].replace(
            "",
            pd.NA
        )

for col in categorical_columns:
    if col in aviation.columns:
        aviation[col] = aviation[col].str.upper()

In [108]:
# Clean number of engines

if "Number.of.Engines" in aviation.columns:
    aviation["Number.of.Engines"] = pd.to_numeric(
        aviation["Number.of.Engines"],
        errors="coerce"
    )

    # Negative engine counts are invalid
    aviation.loc[
        aviation["Number.of.Engines"] < 0,
        "Number.of.Engines"
    ] = np.nan

    display(
        aviation["Number.of.Engines"]
        .value_counts(dropna=False)
        .sort_index()
    )

Number.of.Engines
0.0      753
1.0    55076
2.0     9649
3.0      432
4.0      401
8.0        1
NaN     5005
Name: count, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [109]:
# Calculate missing-value percentages
missing_percent = (
    aviation.isna().mean() * 100
).sort_values(ascending=False)

display(
    missing_percent.to_frame(
        name="Missing_Percentage"
    )
)


# Flag columns with more than 80% missing values
high_missing_columns = missing_percent[
    missing_percent > 80
].index.tolist()

print("Columns with more than 80% missing values:")
print(high_missing_columns)

,Missing_Percentage
Schedule,84.647419
Air.carrier,81.277956
FAR.Description,69.585092
Aircraft.Category,69.296241
Longitude,63.081734
Latitude,63.074723
Airport.Code,43.619333
Airport.Name,40.993031
Broad.phase.of.flight,29.234264
Fatal.Serious.Injury.Rate,17.955046


Columns with more than 80% missing values:
['Schedule', 'Air.carrier']


In [110]:
essential_columns = [
    "Event.Date",
    "Make",
    "Model",
    "Make.Model",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
    "Total.People",
    "Fatal.Serious.Injury.Rate",
    "Aircraft.Damage",
    "Aircraft.Destroyed"
]

# Only drop high-missing columns that are not essential
columns_to_drop = [
    col for col in high_missing_columns
    if col not in essential_columns
]

aviation = aviation.drop(
    columns=columns_to_drop
).copy()

print("Dropped columns:", columns_to_drop)
print("Final cleaned shape:", aviation.shape)

Dropped columns: ['Schedule', 'Air.carrier']
Final cleaned shape: (71317, 34)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [111]:
output_path = data_path / "AviationData_Cleaned.csv"

aviation.to_csv(
    output_path,
    index=False
)

print(f"Cleaned dataset saved to: {output_path}")
print("Final shape:", aviation.shape)

display(aviation.head())

Cleaned dataset saved to: data\AviationData_Cleaned.csv
Final shape: (71317, 34)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.People,Total.Fatal.Serious.Injuries,Fatal.Serious.Injury.Rate,Aircraft.Destroyed,Make.Model
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,NaN,NaN,ONP,NEWPORT MUNICIPAL,...,3.0,VMC,APPROACH,Probable Cause,NaN,4.0,0.0,0.0,0,CESSNA 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,NaN,NaN,3YR,MUNICIPAL,...,2.0,VMC,LANDING,Probable Cause,NaN,2.0,0.0,0.0,0,CESSNA 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,NaN,NaN,78Y,RANKIN,...,1.0,VMC,TAKEOFF,Probable Cause,NaN,1.0,0.0,0.0,0,CESSNA 182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,NaN,NaN,CCB,CABLE,...,0.0,VMC,APPROACH,Probable Cause,NaN,2.0,0.0,0.0,0,PIPER PA-28R-200
3605,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,NaN,NaN,NaN,SPRINGBROOK,...,2.0,VMC,LANDING,Probable Cause,NaN,2.0,0.0,0.0,0,CESSNA 140
